# Load Libraries and Data

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import LogNorm
from shapely.geometry import Point


#Load the cleaned data
clean_rad = pd.read_excel("clean_data_kp.xlsx")

#### Create geodataframe for geopandas

In [ ]:
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

geometry = [Point(xy) for xy in zip(clean_rad["lon"], clean_rad["lat"])]
geo_rad = gpd.GeoDataFrame(clean_rad, geometry=geometry, crs="EPSG:4326")

### Define a constant colormap to use for all visuals

Default: `managua` 

`managua` is very good for highlighting all points on the graph.  
Change to `Reds` if you would rather only easily see extreme high values. All low values become very hard to see on the light background

In [ ]:
COLOR = "managua"

# Create a map for each main particle per second column

In [ ]:
cols = ["proton0_ps", "electron0_ps", "xray0_ps", "total_radiation_ps"]

fig, axes = plt.subplots(2, 2, figsize=(10, 5), constrained_layout=True)

axes = axes.flatten()

for i, col in enumerate(cols):
    world.plot(ax=axes[i], color="lightgrey")
    geo_rad.plot(ax=axes[i],
                markersize=1,
                alpha=0.5,
                column=col, 
                cmap=COLOR,
                legend=True
                )
    axes[i].set_title(f"Variable: {col}")

for ax in axes:  # Fix: loop over each axis
    ax.set_aspect("equal", adjustable="box")

fig.suptitle("All Main Standardized Variables")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad.plot(ax=ax,
            markersize=10,
            alpha=0.5,
            column="xray0_ps", 
            cmap=COLOR,
            legend=True
            )

ax.set_aspect("equal", adjustable="box")
plt.title("Xrays Per Second")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad.plot(ax=ax,
            markersize=10,
            alpha=0.5,
            column="total_radiation_ps", 
            cmap=COLOR,
            legend=True
            )

ax.set_aspect("equal", adjustable="box")
plt.title("Total Radiation Per Second")
plt.show()

## Analysis of particles per second

Comparing all of the low-threshold sensors standardized by second shows some interesting variation. 
- It looks as though, at these sensor thresholds, proton and, to a lesser extent, electeron radiation are higher the further south go. 
- Inversely, x-ray appears to get higher the further north you go. 
    - Does the season have an effect on this stark difference?
    - Is the difference because of hemispheric differece?
- Total radiation per second seems to be fairly steady with the main variation being sample size difference between the arctic and antarctic regions. 
    - The lack of overlap between high proton and electron regions and high xray regions might be why total radiation seems so steady.
    - Interestingly, there is some overlap between the detectors over Asia
- Barring one tiny point in the bottom left of the map, ses_ps is consistently low across the entire planet. That point might be worth researching.

# Split each main particle per second column by month

In [ ]:
months = ["Jan", "Feb", "Mar", "Apr"]
cols = ["proton0_ps", "electron0_ps", "xray0_ps", "ses_ps", "total_radiation_ps"]

for col in cols:
    fig, axes = plt.subplots(2, 2, figsize=(12, 6), constrained_layout=True)
    axes = axes.flatten()

    vmin = geo_rad[col].min()
    vmax = geo_rad[col].max()

    for i, month in enumerate(months):
        subset = geo_rad[geo_rad["month"] == month]
        world.plot(ax=axes[i], color="lightgrey")
        subset.plot(
            ax=axes[i],
            markersize=1,
            alpha=0.5,
            column=col,
            cmap=COLOR,
            legend=False,
            vmin=vmin,
            vmax=vmax
        )
        axes[i].set_title(f"Variable: {month}")
       
    # Create a colorbar with no alha value applied
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    sm = cm.ScalarMappable(cmap=COLOR, norm=norm)
    sm.set_array([])  # required but unused

    plt.colorbar(sm, ax=axes, label=f"{col[:-3].title().replace("_", " ")} Per Second")
    fig.suptitle(f"{col[:-3].title().replace("_", " ")} Per Second by Month", va="top", fontsize="xx-large")
    plt.show()

#### Find the number of data points from each month

In [ ]:
clean_rad.groupby("month", sort=False)["month"].count()

#### Find and view all data points are in the southern hemisphere in March

In [ ]:
print(f"Data points in the entire southern hemisphere in March: {len(clean_rad[(clean_rad["lat"] < 0) & (clean_rad["month"] == "Mar")])}")
print(f"Data points in or below sourthern Temperate Zone in March: {len(clean_rad[(clean_rad["lat"] < -23.5) & (clean_rad["month"] == "Mar")])}")
print(f"Data points in the Antarctic Circle in March: {len(clean_rad[(clean_rad["lat"] < -66.5) & (clean_rad["month"] == "Mar")])}")

In [ ]:
clean_rad[(clean_rad["lat"] < -23.5) & (clean_rad["month"] == "Mar")][["timestamp", "packetID", "sample", "lat", "lon", 
                                                                       "electron0_ps", "proton0_ps", "xray0_ps", "ses_ps",
                                                                       "total_radiation_ps"]]

## Analysis of monthly splitting

#### Splitting these columns by month tells an interesting story:
- Protons stay pretty low for most of the time, but packets of higher counts are more common in January and February and in the southern hemisphere
- Electrons stay fairly steady across the globe, but the few values that get higher are pretty much all in January and February
- As the year progresses from January to April, the X-rays appear to become stronger in the Arctic Circle
    - This seems to correlate with when the sun should rise in the Arctic Circle
        - It might be worth splitting March by week to see if it grows on a weekly basis
    - X-rays stay similarly low in the southern hemisphere over all four months
- Ses basically never varies, and, when it does, it's not by much
    - There is one spike in April near lon -150 lat 60. It might be worth investigating what may have caused that, but Ses is not our focus for this project.
        - Without this one data point, ses's scale caps out at a much lower value
- While we are aware total radiation is not the most useful metric, it might still deliver some good insight.
    - Because of how little overlap between high values in our data, this column shows off every place where `proton0_ps`, `electron0_ps`, and `xray0_ps` spike

#### This brings up a few issues:
- We have almost no data in the southern hemisphere in March
    - In the entire southern hemisphere, we only have 32 data points total
    - Of the 32 data points, only have 17 data points within the South Temperate Zone
        - That accounts for one full packet plus the very first entry of another packet
    - Interestingly, we have more than double the data points in March than any other month despite no data in the south
- We have significantly fewer data points in April
    - This makes sense given our data stops on April 10th
- Only having three and a quarter months of data makes it really hard to make claims relating to seasonality
- Our sample sizes vary wildly based on location and month, making analysis difficult

# View Radiation Per Second in a Log Scale

In [ ]:
clean = geo_rad[geo_rad["xray0_ps"] > 0]
norm = LogNorm(vmin=clean["xray0_ps"].min(), vmax=clean["xray0_ps"].max())

geo_rad['proton_electron'] = geo_rad["proton0_ps"] + geo_rad["electron0_ps"]

cols = [("proton0_ps", "Proton 0"), ("electron0_ps", "Electron 0"), ("proton_electron", "Protons + Electrons"), 
        ("xray0_ps", "X-ray 0"), ("total_radiation_ps", "Total Radiation")]

for column, title in cols:
    fig, ax = plt.subplots(figsize=(14, 6))

    world.plot(ax=ax, color="lightgrey")
    geo_rad.plot(ax=ax,
                column=column,
                cmap=COLOR,
                norm=norm,
                alpha=0.5,
                markersize=10,
                legend=True,
                legend_kwds={"label": f"{title} Per Second (Log Scale)"}
    )

    plt.title(f"{title} Per Second (Log Scale)")
    plt.show()

## Analysis of Log Scales

The biggest thing these highlight is that there appears to be a minor hotspot over Asia. All five graphs show elevated levels here under a logarithmic scale. Nathan Turner theorized that this could be some sort of weakspot in Earth's magnetic field since it's about opposite to the South Atlantic Anomoly. This is definetely worth investigating further
  
This log scale also shows that proton0_ps does have elevated levels in the Arctic Circle in such a way that is hard to see outside of a logarithmic scale